# 01 Preprocess: Stages 1-7

Thin entry point that runs preprocessing, OCR, and text classification on a single SIP PDF. All real logic lives in the `sip_extractor` package; this notebook just wires together inputs, outputs, and inline previews.

Runs locally on CPU. Stages 1-5 take ~10-30s, Stage 6 (PaddleOCR) takes 30-60s on first run (model download), faster afterwards.

## Setup

Detects whether we are on Colab or local. On Colab, mounts Drive and points the model caches at `.model_cache/` so PaddleOCR / DINOv2 / Qwen don't redownload every session. Locally, `PROJECT_ROOT` is the repo root.

In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/sip-extractor")
    cache = PROJECT_ROOT / ".model_cache"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache / "huggingface")
    os.environ["TORCH_HOME"] = str(cache / "torch")
    os.environ["PADDLE_HOME"] = str(cache / "paddle")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working dir: {PROJECT_ROOT}")
print(f"Colab: {IS_COLAB}")

## Configure inputs and outputs

Point `PDF_PATH` at the SIP you want to process. Outputs go to `outputs/<sip_stem>/`.

In [ ]:
PDF_PATH = PROJECT_ROOT / "data" / "sips" / "MUNDEWADI-SIP.pdf"
TARGET_DPI = 300
OUT_DIR = PROJECT_ROOT / "outputs" / PDF_PATH.stem

if not PDF_PATH.exists():
    legacy = PROJECT_ROOT / "MUNDEWADI-Signaling_Interlocking_Plan__SIP__from_Railways.pdf"
    if legacy.exists():
        PDF_PATH = legacy
        OUT_DIR = PROJECT_ROOT / "outputs" / PDF_PATH.stem
        print(f"Using legacy PDF location: {PDF_PATH}")

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"Input : {PDF_PATH}")
print(f"Output: {OUT_DIR}")

## Stages 1-5: Preprocessing

Render at target DPI, drop colored markup, Sauvola binarize, crop title blocks off the sides, save.

In [ ]:
from IPython.display import Image as IPyImage
from sip_extractor import preprocessing

prep = preprocessing.run(PDF_PATH, OUT_DIR, target_dpi=TARGET_DPI)
print(f"Crop x=[{prep.crop_x_min}, {prep.crop_x_max}]")
print(f"Binary    : {prep.binary_path}")
print(f"Preview   : {prep.binary_preview_path}")
IPyImage(prep.binary_preview_path)

## Stage 6: OCR

Tile-based PaddleOCR over the cleaned grayscale (NOT the binary). First run downloads ~few hundred MB of models, cached under `~/.paddleocr/` (or `PADDLE_HOME` on Colab).

In [ ]:
from sip_extractor import ocr

text_entities = ocr.run(prep.gray_cropped, OUT_DIR)
print(f"Detected {len(text_entities)} unique text regions")
IPyImage(str(OUT_DIR / "text_overlay_preview.png"))

## Stage 7: Text classification

Bucket each detection into `signal_id`, `track_circuit`, `km_marker`, `track_label`, `point_id`, `dimension`, or `note`. Rewrites `text.json` with the `category` field added.

In [ ]:
from sip_extractor import classify

classify.run(text_entities, OUT_DIR, gray_cropped=prep.gray_cropped)

counts = classify.category_counts(text_entities)
print("Category breakdown:")
for cat, n in counts.most_common():
    print(f"  {cat:15s}: {n}")

IPyImage(str(OUT_DIR / "text_categorized_preview.png"))

## Outputs

After this notebook runs, `outputs/<sip_stem>/` contains:

- `binary.png` — full-resolution cropped binary
- `binary_preview.png` — 3000px-wide preview
- `text.json` — detections with `text`, `text_normalized`, `score`, `bbox`, `category`
- `text_overlay.png` / `text_overlay_preview.png` — raw OCR overlay
- `text_categorized_overlay.png` / `text_categorized_preview.png` — color-coded by category

Stage 8 (track detection) is broken and deferred until after Stage 9 (symbol detection). See HANDOFF.md.